In [ ]:
# Save v4's held-out predictions (for the v4 + cross-encoder blend). Run in the Step 1 v4 session.
if not all((CDIR[c] / "pids.npy").exists() for c in COUNTRIES):      # unpacked copies removed by clean-up
    CDIR = {c: d for c, d in by_country("qids.npy", "ckpt_train").items() if (d / "pairs.npz").exists()}
rows = []
for c in COUNTRIES:
    r = R[c]
    m = is_val(r["b"])
    pids = np.load(CDIR[c] / "pids.npy", allow_pickle=True)
    prob = matcher.predict(r["X"][m], num_iteration=matcher.best_iteration).astype(np.float32)
    rows.append(pd.DataFrame({"country": c, "s1": META[c]["qids"][r["q"][m]], "cand": pids[r["p"][m]],
                              "prob": prob, "y": r["y"][m]}))
H4 = pd.concat(rows, ignore_index=True)
H4.to_parquet(WORK / "heldout_v4.parquet", index=False)
print(f"saved {WORK / 'heldout_v4.parquet'}: {len(H4):,} held-out pairs, {H4['s1'].nunique():,} entities")